# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [30]:
#Imports
import os
import requests
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

In [31]:
#Load API Keys and Create Clients
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OLLAMA_BASE_URL = "http://localhost:11434/v1"

openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

ollama_client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

In [32]:
#Model Options
MODEL_OPTIONS = {
    "GPT (OpenRouter)": {
        "client": openrouter_client,
        "model": "gpt-4.1-mini"
    },
    "Llama (Ollama)": {
        "client": ollama_client,
        "model": "llama3.2"
    }
}

In [33]:
#System Prompt
SYSTEM_PROMPT = """
You are an expert AI tutor helping students understand artificial intelligence
and machine learning concepts.

Rules:
- Explain ideas simply
- Use examples when helpful
- Avoid unnecessary jargon
- Keep explanations concise
"""

In [34]:
#Wikipedia Lookup tool
def wikipedia_lookup(query):
    
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query}"

    response = requests.get(url)

    if response.status_code != 200:
        return "No Wikipedia summary available."

    data = response.json()

    return data.get("extract", "")

In [35]:
#Build Messages with Chat History
def build_messages(history, user_question):

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for msg in history:
        messages.append(msg)

    messages.append({"role": "user", "content": user_question})

    return messages

In [36]:
#Stream Response
def stream_response(history, user_question, model_choice):

    config = MODEL_OPTIONS[model_choice]

    client = config["client"]
    model = config["model"]

    wiki_info = wikipedia_lookup(user_question)

    prompt = f"""
{user_question}

Reference information from Wikipedia:
{wiki_info}
"""

    messages = build_messages(history, prompt)

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    reply = ""

    for chunk in stream:

        if not chunk.choices:
            continue

        delta = chunk.choices[0].delta

        if not delta or not delta.content:
            continue

        reply += delta.content
        yield reply

In [37]:
#Chat Function
def chat_fn(user_message, history, model_choice):

    history = history or []

    history.append({"role": "user", "content": user_message})

    assistant_message = {"role": "assistant", "content": ""}

    for partial in stream_response(history, user_message, model_choice):
        assistant_message["content"] = partial
        yield history + [assistant_message]

    history.append(assistant_message)

In [38]:
#Gradio UI
with gr.Blocks() as demo:

    gr.Markdown("# AI Study Companion")
    gr.Markdown("Ask questions about AI concepts like transformers, reinforcement learning, or vector embeddings.")

    with gr.Row():
        model_choice = gr.Dropdown(
            choices=list(MODEL_OPTIONS.keys()),
            value="GPT (OpenRouter)",
            label="Model"
        )

    chatbot = gr.Chatbot()

    msg = gr.Textbox(
        placeholder="Ask an AI concept...",
        label="Your Question"
    )

    send_btn = gr.Button("Ask")

    state = gr.State([])

    send_btn.click(
        chat_fn,
        inputs=[msg, state, model_choice],
        outputs=chatbot
    )

In [39]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
